ej 1

In [1]:
import openeo
conn = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


ej 2 y 3
de enero a agosto 2025

In [13]:
COLLECTION = "SENTINEL2_L1C"                 
TEMPORAL_EXTENT = ["2025-01-01", "2025-08-14"]
MAX_CLOUD = 20

BANDS = ["B02", "B03", "B04", "B05", "B07", "B8A", "B08", "B11", "B12"]

# BBoxes (EPSG:4326)
lago_atitlan = {"west": -91.326256, "east": -91.07151, "south": 14.5948,   "north": 14.750979}
lago_amatitlan = {"west": -90.638065, "east": -90.512924, "south": 14.412347, "north": 14.493799}

def to_extent(box):
    return {"west": box["west"], "south": box["south"], "east": box["east"], "north": box["north"], "crs": "EPSG:4326"}

def descargar_lago(nombre, box, carpeta):
    print(f"\n🔎 {nombre}: S2 L1C {TEMPORAL_EXTENT[0]} → {TEMPORAL_EXTENT[1]} con ≤{MAX_CLOUD}% nubes, remuestreado a 10 m (EPSG:32616)…")

    cube = conn.load_collection(
        collection_id=COLLECTION,
        spatial_extent=to_extent(box),
        temporal_extent=TEMPORAL_EXTENT,
        bands=BANDS,
        max_cloud_cover=MAX_CLOUD
    )

    # remuestreo a 10 m y proyección común (UTM 16N ~ Guatemala)
    cube = cube.resample_spatial(resolution=10, projection="EPSG:32616", method="near")

    # GeoTIFF multibanda por fecha (time-slices)
    result = cube.save_result(format="GTIFF")
    job = conn.create_job(result, title=f"{nombre} S2 L1C ≤{MAX_CLOUD}% (ene-ago 2025, 10m)")
    job.start_and_wait().get_results().download_files(target=carpeta)

    print(f"✅ {nombre}: archivos guardados en './{carpeta}'")


descargar_lago("Lago Atitlán",   lago_atitlan,   "descargas_atitlan")
descargar_lago("Lago Amatitlán", lago_amatitlan, "descargas_amatitlan")


🔎 Lago Atitlán: S2 L1C 2025-01-01 → 2025-08-14 con ≤20% nubes, remuestreado a 10 m (EPSG:32616)…
0:00:00 Job 'j-250815053209400fa5e3d5c68b43e1c0': send 'start'


KeyboardInterrupt: 